In [1]:
from pathlib import Path
import pandas as pd
import re
import html
import unicodedata

In [ ]:
# A função clean_snippet é projetada para limpar e normalizar o texto de um snippet, removendo HTML, URLs, caracteres de controle e outros elementos indesejados, resultando em um texto mais limpo e legível.
def clean_snippet(text: str) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    s = html.unescape(s)
    s = s.replace("\u00a0", " ")  # non-breaking space
    s = unicodedata.normalize("NFC", s)
    s = re.sub(r"<[^>]+>", " ", s)  # drop HTML tags
    s = re.sub(r"https?://\S+|www\.\S+", " ", s)  # drop URLs
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"[\x00-\x1f\x7f-\x9f]+", " ", s)  # control chars
    s = re.sub(r"[•·▪●■◆▶▲►]+", " ", s)  # bullet-like glyphs
    s = re.sub(r"\s+", " ", s).strip()
    return s
# O padrão de localização procura por preposições comuns seguidas de um possível nome de local, que deve começar com letra maiúscula e não conter pontuação ou dígitos. O nome do local é então limpo para remover conjunções e caracteres de separação, mantendo apenas a parte mais relevante.
location_pattern = re.compile(
    r"\b(?:em|no|na|nos|nas|perto de|junto a)\s+([A-ZÁÂÃÀÉÊÍÓÔÕÚÇ][^\.,;:()\[\]{}\-\d]{1,60})",
    re.IGNORECASE,
)
# A função extract_location_from_snippet utiliza o padrão de localização para extrair o nome do local de um snippet limpo. Se o texto for nulo ou não contiver uma correspondência, retorna uma string vazia. Caso contrário, extrai o nome do local, limpa-o de conjunções e caracteres de separação, e retorna a parte mais relevante do nome do local.
def extract_location_from_snippet(text: str) -> str:
    if pd.isna(text):
        return ""
    s = str(text)
    match = location_pattern.search(s)
    if not match:
        return ""
    loc = match.group(1).strip()
    loc = re.split(r"\s+(?:e|ou)\s+|\s+-\s+|\s+—\s+|\s+–\s+", loc)[0].strip()
    return loc

def read_csv_safe(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=None, engine="python", on_bad_lines="skip")

def clean_dataframe(df_local: pd.DataFrame) -> pd.DataFrame:
    clean_local = df_local.copy()

    for col in ["query", "title", "snippet", "original_url", "archive_url"]:
        if col in clean_local.columns:
            clean_local[col] = clean_local[col].astype(str).str.strip().replace("nan", "")

    if {"original_url", "archive_url"}.issubset(clean_local.columns):
        clean_local = clean_local[~((clean_local["original_url"] == "") & (clean_local["archive_url"] == ""))].copy()

    if "archive_date" in clean_local.columns:
        clean_local["archive_date"] = clean_local["archive_date"].astype(str).str.strip()
        clean_local["archive_date_dt"] = pd.to_datetime(
            clean_local["archive_date"], format="%Y%m%d%H%M%S", errors="coerce"
        )

    if "year" in clean_local.columns:
        clean_local["year"] = pd.to_numeric(clean_local["year"], errors="coerce")
    elif "archive_date_dt" in clean_local.columns:
        clean_local["year"] = clean_local["archive_date_dt"].dt.year

    if {"archive_url", "archive_date"}.issubset(clean_local.columns):
        clean_local = clean_local.drop_duplicates(subset=["archive_url", "archive_date"])
    elif "archive_url" in clean_local.columns:
        clean_local = clean_local.drop_duplicates(subset=["archive_url"])

    if "snippet" in clean_local.columns:
        clean_local["snippet_clean"] = clean_local["snippet"].map(clean_snippet)
        clean_local = clean_local.drop(columns=["snippet"])

    if "title" in clean_local.columns:
        clean_local = clean_local.drop_duplicates(subset=["title"])

    if "snippet_clean" in clean_local.columns:
        clean_local["location_name"] = clean_local["snippet_clean"].map(extract_location_from_snippet)
        clean_local["location_source"] = clean_local["location_name"].apply(
            lambda x: "snippet_clean" if x else ""
        )

    if "location_name" in clean_local.columns:
        clean_local = clean_local[clean_local["location_name"] != ""].copy()

    return clean_local
# A função extract_keyword é projetada para extrair uma palavra-chave de um caminho de arquivo. Ela divide o nome do arquivo (sem extensão) em partes usando o caractere de sublinhado como separador. Se as últimas duas partes indicarem um formato específico (uma letra "c" e um número), a função retorna a parte anterior como a palavra-chave. Caso contrário, retorna o nome do arquivo completo como a palavra-chave.
def extract_keyword(path: Path) -> str:
    parts = path.stem.split("_")
    if len(parts) >= 3 and parts[-1].lower() == "c" and parts[-2].isdigit():
        return "_".join(parts[:-2])
    return path.stem

In [ ]:
data_dir = Path("dados_arquivopt")
output_dir = Path("clean_data")
output_dir.mkdir(exist_ok=True)

grouped = {}
for path in sorted(data_dir.glob("*_C.csv")):
    df = read_csv_safe(path)
    clean_df = clean_dataframe(df)
    clean_df["source_file"] = path.name
    key = extract_keyword(path)
    grouped.setdefault(key, []).append(clean_df)

saved_files = []
for key, dfs in grouped.items():
    combined = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    out_path = output_dir / f"{key}_clean.csv"
    combined.to_csv(out_path, index=False)
    saved_files.append(out_path)

saved_files

[WindowsPath('clean_data/abandono_rural_clean.csv'),
 WindowsPath('clean_data/desenvolvimento_rural_clean.csv'),
 WindowsPath('clean_data/programa_de_apoio_ao_interior_clean.csv')]